# LatentFed — full method

**Self-contained Kaggle notebook.** All models, losses, local updates, server operations and experiment execution are ordinary Python cells in this file.

Enable **Internet** and a **T4 GPU**, then choose **Run All**. The setup keeps the platform's compatible torch/torchvision pair. If numerical packages are updated after import, restart the kernel once and Run All.

Defaults: CIFAR-10 and EMNIST Balanced; three Dirichlet concentrations; five seeds; **20 clients, 20 participants per round, 100 rounds, one local epoch and batch size 32**.

1. Runtime and experiment selection
2. Configuration, data and reproducible partitions
3. Classifier and autoencoder, MDFE and CALD
4. Local training, server updates and evaluation
5. Federated loop, checkpoints, results and download

Scientific operations come from the supplied LatentFed final revision notebook. Execution is serial on one device. No repository clone, source folder or account token is required.

## 1. Dependencies

Install numerical dependencies before importing the method.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util
import importlib.metadata

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
for package in ("torch", "torchvision"):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError("Select a Kaggle GPU runtime with torch and torchvision, then restart.")
dependencies = [
    ("numpy", "numpy", "2.1.3"),
    ("pandas", "pandas", "2.3.3"),
    ("scipy", "scipy", "1.16.3"),
    ("sklearn", "scikit-learn", "1.6.1"),
    ("hdbscan", "hdbscan", "0.8.44"),
]
install, restart_packages = [], []
for module, distribution, version in dependencies:
    try:
        current_version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        install.append(f"{distribution}=={version}")
        if module in sys.modules:
            restart_packages.append(module)
for module, requirement in [("matplotlib", "matplotlib>=3.9,<3.11"), ("tqdm", "tqdm>=4.66,<5")]:
    if importlib.util.find_spec(module) is None:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])
if restart_packages:
    raise RuntimeError(
        "Dependencies were updated. Restart the kernel and Run All: " + ", ".join(restart_packages)
    )
print("Dependencies ready. All method code is contained in the cells below.")


## 2. Imports

Only standard Python and installed scientific packages are imported.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import random
import time
import types
import zipfile
from dataclasses import asdict, dataclass, replace
from contextlib import contextmanager
from typing import Any
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
)
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from tqdm.auto import tqdm
from IPython.display import display, FileLink
import hdbscan
import matplotlib.pyplot as plt

PAPER_DATASETS = ("cifar10", "emnist")
PAPER_ALPHAS = (0.3, 0.5, 0.7)
PAPER_SEEDS = (42, 101, 2024, 3407, 8888)
SUPPORTED_VARIANTS = ("full", "no_mdfe", "no_cald", "ae_only")
DEV_SEEDS = [7, 19]
FINAL_SEEDS = list(PAPER_SEEDS)
FAIR_PROTOCOL_VERSION = "neurocomp-fair-v1"
COMMON_PROTOCOL_SHA256 = "445d12940d051634680727dc3739822c4030a90d7a3a3a90894f1d42e490d2fe"
METHOD_IMPLEMENTATION_VERSION = "latentfed-standalone-v2"
CALD_REVISION_VERSION = METHOD_IMPLEMENTATION_VERSION


## 3. Select the experiment batch

Select official seeds and concentrations here. A smaller real-data batch still uses 100 rounds per experiment. `SMOKE_TEST=True` uses synthetic data and two rounds; those results are excluded from paper statistics.

In [ ]:
DATASETS = ["cifar10", "emnist"]
ALPHAS = [0.3, 0.5, 0.7]
SEEDS = [42, 101, 2024, 3407, 8888]
VARIANTS = ["full"]
METHOD = "latentfed"
SMOKE_TEST = os.environ.get("LATENTFED_SMOKE", "0") == "1"
RESUME = True
INCLUDE_CHECKPOINTS_IN_ZIP = False
default_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT = Path(os.environ.get("LATENTFED_OUTPUT_ROOT", str(default_root / "01_latentfed_full")))
print("Mode:", "synthetic smoke check" if SMOKE_TEST else "real-data paper grid")
print("Outputs:", OUTPUT_ROOT)


## 4. Experiment configuration

The budget and method parameters are explicit. Evaluation uses the official test split.

In [ ]:
@dataclass
class ExperimentConfig:
    dataset: str = "cifar10"
    alpha: float | None = 0.3
    method: str = "latentfed_fedavg"
    seed: int = 42
    num_clients: int = 20
    participation_ratio: float = 1.0
    rounds: int = 100
    local_epochs: int = 1
    batch_size: int = 32
    num_workers: int = 2
    latent_dim: int = 128
    classifier_lr_cifar10: float = 0.0005
    classifier_lr_emnist: float = 0.0001
    autoencoder_lr: float = 0.0005
    weight_decay: float = 0.0001
    lambda_rec: float = 1.0
    lambda_str: float = 0.05
    lambda_inv: float = 0.05
    lambda_clu: float = 0.05
    lambda_margin: float = 0.05
    margin_cls: float = 1.0
    margin_clu: float = 1.0
    fedprox_mu: float = 0.01
    cald_warmup_rounds: int = 20
    cald_ramp_rounds: int = 20
    mdfe_mode: str = "full"
    cald_mode: str = "fixed_split"
    cald_target: str = "sample"
    invariant_ratio: float = 0.5
    router_temperature: float = 1.0
    router_init_logit: float = 2.0
    lambda_route_soft: float = 0.001
    lambda_route_budget: float = 1.0
    router_min_fraction: float = 0.25
    router_max_fraction: float = 0.75
    summary_view: str = "deterministic"
    exclude_noise_from_prototypes: bool = True
    dp_noise_std: float = 0.0
    cluster_min_size: int = 2
    checkpoint_every: int = 5
    cluster_geometry: str = "full"
    cluster_alignment: str = "greedy"
    alignment_max_cosine_distance: float = 1.0
    prototype_ema_beta: float = 0.0
    summary_ema_beta: float = 0.0
    mdfe_rho_mode: str = "positive"
    ae_pretrain_rounds: int = 0
    cluster_min_samples: int = 1
    data_root: str = "./data"
    validation_fraction: float = 0.1
    min_train_samples: int = 10
    min_validation_samples: int = 1
    min_test_samples: int = 1
    partition_max_attempts: int = 100
    evaluation_split: str = "test"
    allow_custom_protocol: bool = False
    variant: str = "full"
    ae_architecture: str = "convolutional"
    fc_hidden_dim: int = 512

    @property
    def uses_latentfed(self) -> bool:
        return self.method.startswith("latentfed")

    @property
    def uses_fedprox(self) -> bool:
        return self.method in {"fedprox", "latentfed_fedprox"}

    @property
    def classifier_lr(self) -> float:
        return (
            self.classifier_lr_cifar10 if self.dataset == "cifar10" else self.classifier_lr_emnist
        )

    def validate(self):
        validate_common_config(self)
        if self.cluster_geometry not in {"full", "cluster_only"} or self.cluster_alignment not in {
            "greedy",
            "hungarian",
        }:
            raise ValueError("Invalid geometry/alignment")
        if self.mdfe_rho_mode not in {"original", "positive"}:
            raise ValueError("Invalid rho mode")
        if not all((0 <= b < 1 for b in (self.prototype_ema_beta, self.summary_ema_beta))):
            raise ValueError("EMA beta must be in [0, 1)")
        if not 0 <= self.ae_pretrain_rounds < self.rounds:
            raise ValueError("Warm-up must fit inside the fixed AE update budget")
        if self.cald_mode == "fixed_split" and (self.latent_dim % 2 or self.invariant_ratio != 0.5):
            raise ValueError("Paper fixed split requires equal halves")
        if (
            self.cluster_min_size < 2
            or self.cluster_min_samples < 1
            or (not 0 <= self.alignment_max_cosine_distance <= 1)
        ):
            raise ValueError("Invalid clustering parameters")
        if self.mdfe_mode not in {
            "full",
            "off",
            "mean_only",
            "peak_only",
            "mean_peak",
            "fixed_distortion",
            "std_distortion",
        }:
            raise ValueError("Unknown mdfe_mode")
        if self.cald_mode not in {"adaptive", "fixed_split", "fixed_soft", "legacy", "off"}:
            raise ValueError("Unknown cald_mode")
        if self.cald_target not in {"sample", "batch_mean"}:
            raise ValueError("cald_target must be sample or batch_mean")
        if self.summary_view not in {"train", "deterministic"}:
            raise ValueError("summary_view must be train or deterministic")
        if (
            self.latent_dim < 2
            or not 0 < self.invariant_ratio < 1
            or (not 0 < self.invariant_dim < self.latent_dim)
        ):
            raise ValueError("Both latent branches must have at least one coordinate")
        if self.cald_mode == "legacy" and (
            self.latent_dim % 2 or self.invariant_ratio != 0.5 or self.cald_target != "sample"
        ):
            raise ValueError("legacy requires an even latent_dim, ratio 0.5 and sample targets")
        if self.router_temperature <= 0 or self.router_init_logit <= 0:
            raise ValueError("Router temperature and initial logit magnitude must be positive")
        if not 0 < self.router_min_fraction < self.router_max_fraction < 1:
            raise ValueError("Invalid routing budget range")
        if (
            min(
                self.lambda_route_soft,
                self.lambda_route_budget,
                self.dp_noise_std,
                self.cald_warmup_rounds,
                self.cald_ramp_rounds,
            )
            < 0
        ):
            raise ValueError("Loss weights, noise and curriculum durations must be nonnegative")
        if (
            not 0 < self.participation_ratio <= 1
            or min(
                self.rounds,
                self.local_epochs,
                self.batch_size,
                self.num_clients,
                self.checkpoint_every,
            )
            < 1
        ):
            raise ValueError("Invalid training budget or participation")
        if self.uses_latentfed and hdbscan is None:
            raise RuntimeError(
                "HDBSCAN is required. Install hdbscan and restart; no silent DBSCAN substitution is allowed."
            )
        if self.variant not in SUPPORTED_VARIANTS:
            raise ValueError(f"Unknown variant: {self.variant}")
        if (
            self.ae_architecture not in {"convolutional", "fully_connected"}
            or self.fc_hidden_dim < 1
        ):
            raise ValueError("Invalid autoencoder architecture")
        if self.num_clients != 20 or self.participation_ratio != 1.0:
            raise ValueError("The release uses 20 clients and all 20 participate in every round.")
        if not self.allow_custom_protocol and (self.rounds != 100 or self.seed not in PAPER_SEEDS):
            raise ValueError("Paper mode requires 100 rounds and one of the five official seeds.")
        if self.variant == "ae_only" and (
            self.ae_architecture != "fully_connected"
            or self.mdfe_mode != "off"
            or self.cald_mode != "off"
        ):
            raise ValueError("AE only requires a fully connected AE with MDFE/CALD disabled.")
        if self.variant != "full" or self.ae_architecture != "convolutional":
            raise ValueError(
                "Use the separate ablation notebook for module variants and the fully connected AE."
            )

    @property
    def invariant_dim(self):
        return int(round(self.latent_dim * self.invariant_ratio))

    def set_scenario(self, scenario):
        profiles = {"20_full": (20, 1.0), "100_random10": (100, 0.1)}
        if scenario not in profiles:
            raise ValueError(f"Unknown scenario: {scenario}")
        self.num_clients, self.participation_ratio = profiles[scenario]
        return self

    @property
    def config_id(self):
        excluded = {"data_root", "checkpoint_every", "num_workers"}
        payload = {k: v for k, v in asdict(self).items() if k not in excluded}
        payload.update(
            protocol_version=FAIR_PROTOCOL_VERSION,
            common_code=COMMON_PROTOCOL_SHA256,
            method_code=METHOD_IMPLEMENTATION_VERSION,
        )
        return hashlib.sha256(
            json.dumps(payload, sort_keys=True, allow_nan=False).encode()
        ).hexdigest()[:12]

    @property
    def run_name(self):
        dist = "iid" if self.alpha is None else f"alpha_{self.alpha}"
        return f"{self.dataset}_{dist}_{self.method}_n{self.num_clients}_p{self.participation_ratio:g}_{self.evaluation_split}_seed{self.seed}_{self.config_id}"


## 5. Seeded random streams

Role-specific streams pair data, initialization and minibatches across methods.

In [ ]:
def set_seed(seed: int) -> None:
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def state_dict_to_cpu(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in state.items()}


def weighted_average_states(
    states: list[dict[str, torch.Tensor]], weights: list[float]
) -> dict[str, torch.Tensor]:
    total = float(np.sum(weights))
    if total <= 0:
        weights = [1.0 / len(states)] * len(states)
    else:
        weights = [float(w) / total for w in weights]
    out: dict[str, torch.Tensor] = {}
    for key in states[0].keys():
        if not torch.is_floating_point(states[0][key]):
            out[key] = states[0][key].detach().cpu().clone()
            continue
        acc = None
        for state, weight in zip(states, weights):
            value = state[key].detach().float().cpu() * weight
            acc = value if acc is None else acc + value
        out[key] = acc
    return out


def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)


def role_seed(seed, *roles):
    value = json.dumps([int(seed), *roles], separators=(",", ":"))
    return int.from_bytes(hashlib.sha256(value.encode()).digest()[:4], "little")


@contextmanager
def cpu_random_stream(seed):
    """Isolate augmentation/model construction from other algorithm activity."""
    python_state, numpy_state = (random.getstate(), np.random.get_state())
    try:
        with torch.random.fork_rng(devices=[]):
            random.seed(seed)
            np.random.seed(seed)
            torch.random.default_generator.manual_seed(seed)
            yield
    finally:
        random.setstate(python_state)
        np.random.set_state(numpy_state)


def array_hash(arrays):
    digest = hashlib.sha256()
    for key, value in sorted(arrays.items()):
        a = np.ascontiguousarray(value)
        digest.update(key.encode())
        digest.update(str(a.dtype).encode())
        digest.update(json.dumps(a.shape).encode())
        digest.update(a.tobytes())
    return digest.hexdigest()


def common_selection(cfg, round_idx):
    count = max(1, int(round(cfg.num_clients * cfg.participation_ratio)))
    rng = np.random.default_rng(role_seed(cfg.seed, "participants", cfg.num_clients, round_idx))
    return sorted(rng.choice(cfg.num_clients, size=count, replace=False).tolist())


def validate_common_config(cfg):
    if not cfg.allow_custom_protocol:
        if (cfg.num_clients, cfg.participation_ratio) not in {(20, 1.0), (100, 0.1)}:
            raise ValueError(
                "Use set_scenario('20_full') or set_scenario('100_random10'); custom studies require allow_custom_protocol=True"
            )
        expected = {
            "local_epochs": 1,
            "batch_size": 32,
            "classifier_lr_cifar10": 0.0005,
            "classifier_lr_emnist": 0.0001,
            "weight_decay": 0.0001,
        }
        for key, value in expected.items():
            if getattr(cfg, key) != value:
                raise ValueError(
                    f"Common benchmark requires {key}={value}; declare a custom protocol for another setting"
                )
        if getattr(cfg, "local_optimizer", "adam").lower() != "adam":
            raise ValueError("The common classifier optimizer is Adam")
    if getattr(cfg, "summary_view", "deterministic") != "deterministic":
        raise ValueError("The common protocol requires a deterministic training summary view")
    if cfg.dataset not in {"cifar10", "emnist"} or cfg.evaluation_split not in {
        "validation",
        "test",
    }:
        raise ValueError("Unknown dataset/evaluation split")
    if not 0 < cfg.validation_fraction < 1 or not 0 < cfg.participation_ratio <= 1:
        raise ValueError("Invalid validation or participation fraction")
    if cfg.alpha is not None and (not np.isfinite(cfg.alpha) or cfg.alpha <= 0):
        raise ValueError("Dirichlet alpha must be positive or None for IID")
    if (
        min(
            cfg.num_clients,
            cfg.rounds,
            cfg.batch_size,
            cfg.local_epochs,
            cfg.partition_max_attempts,
            cfg.min_train_samples,
            cfg.min_validation_samples,
            cfg.min_test_samples,
            cfg.checkpoint_every,
        )
        < 1
    ):
        raise ValueError("Training budgets and partition minimums must be positive")
    if (
        cfg.participation_ratio < 1
        and getattr(cfg, "reassign_all_clients_each_round", False)
        and (cfg.method in {"ifca", "hypcluster"})
    ):
        raise ValueError(
            "Loss-based reassignment may contact only selected clients in the partial-participation protocol"
        )


## 6. Dataset loading

CIFAR-10 uses a pinned HF archive and archive/member checksums. EMNIST is the 47-class Balanced split with orientation correction. Downloads are cached; Add Input and credentials are unnecessary.

In [ ]:
CIFAR10_HF_REPO = "MIT-OL-AI-D/cifar-10-python"

CIFAR10_HF_REVISION = "a48007227f9e2cd0af96175f4afb5ac4965e261b"

CIFAR10_HF_URL = f"https://huggingface.co/datasets/{CIFAR10_HF_REPO}/resolve/{CIFAR10_HF_REVISION}/cifar-10-python.tar.gz"

CIFAR10_ARCHIVE_MD5 = "c58f30108f718f92721af3b95e74349a"


class EMNISTFixOrientation:

    def __call__(self, img):
        return TF.hflip(TF.rotate(img, -90))


def download_cifar10(data_root):
    """Download the original archive from a pinned HF mirror, retaining sample order."""
    from torchvision.datasets import CIFAR10
    from torchvision.datasets.utils import check_integrity, download_and_extract_archive

    root = Path(data_root).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    files = [
        *CIFAR10.train_list,
        *CIFAR10.test_list,
        (CIFAR10.meta["filename"], CIFAR10.meta["md5"]),
    ]

    def verified():
        return all((check_integrity(root / CIFAR10.base_folder / name, md5) for name, md5 in files))

    if not verified():
        print(f"Downloading/verifying CIFAR-10 from Hugging Face: {CIFAR10_HF_URL}")
        download_and_extract_archive(
            CIFAR10_HF_URL, str(root), filename="cifar-10-python.tar.gz", md5=CIFAR10_ARCHIVE_MD5
        )
        if not verified():
            raise RuntimeError("CIFAR-10 mirror files failed torchvision integrity verification")
    return str(root)


def get_dataset_bundle(dataset_name: str, data_root: str = "./data"):
    dataset_name = dataset_name.lower()
    if dataset_name == "cifar10":
        train_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.2435, 0.2616)),
            ]
        )
        test_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.2435, 0.2616)),
            ]
        )
        cifar_root = download_cifar10(data_root)
        train_ds = datasets.CIFAR10(
            root=cifar_root, train=True, download=False, transform=train_transform
        )
        test_ds = datasets.CIFAR10(
            root=cifar_root, train=False, download=False, transform=test_transform
        )
        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)
        print(
            f"CIFAR-10 ready from verified Hugging Face cache at {cifar_root}: {len(train_ds):,} train / {len(test_ds):,} test"
        )
        return (train_ds, test_ds, labels_train, labels_test, 3, 32, 10)
    if dataset_name == "emnist":
        train_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        test_transform = transforms.Compose(
            [
                EMNISTFixOrientation(),
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )
        train_ds = datasets.EMNIST(
            root=data_root, split="balanced", train=True, download=True, transform=train_transform
        )
        test_ds = datasets.EMNIST(
            root=data_root, split="balanced", train=False, download=True, transform=test_transform
        )
        labels_train = np.asarray(train_ds.targets)
        labels_test = np.asarray(test_ds.targets)
        return (train_ds, test_ds, labels_train, labels_test, 1, 28, 47)
    raise ValueError(f"Unsupported dataset: {dataset_name}")


## 7. Matched client partitions

One class-to-client Dirichlet allocation is shared by official train/test splits. Approximately 10% of each client's training classes is held out. Coverage, disjointness and client minimums are checked.

In [ ]:
def allocate_counts(count, probabilities):
    expected = count * probabilities
    counts = np.floor(expected).astype(np.int64)
    order = np.argsort(-(expected - counts), kind="stable")
    counts[order[: count - counts.sum()]] += 1
    return counts


def make_common_partitions(train_labels, test_labels, cfg):
    """One class-to-client probability matrix for both official data splits."""
    train_labels = np.asarray(train_labels, dtype=np.int64)
    test_labels = np.asarray(test_labels, dtype=np.int64)
    classes = np.unique(train_labels)
    if (
        not np.array_equal(classes, np.arange(len(classes)))
        or not np.isin(test_labels, classes).all()
    ):
        raise ValueError(
            "Labels must be contiguous zero-based classes, with test classes present in training"
        )
    if (
        len(train_labels) < cfg.num_clients * (cfg.min_train_samples + cfg.min_validation_samples)
        or len(test_labels) < cfg.num_clients * cfg.min_test_samples
    ):
        raise ValueError("Dataset is too small for the requested client minimums")
    n = cfg.num_clients
    for attempt in range(cfg.partition_max_attempts):
        rng = np.random.default_rng(role_seed(cfg.seed, "allocation", attempt))
        probabilities = (
            np.full((len(classes), n), 1 / n)
            if cfg.alpha is None
            else rng.dirichlet(np.full(n, cfg.alpha), size=len(classes))
        )
        groups = {split: [[] for _ in range(n)] for split in ("train", "validation", "test")}
        for cls in classes:
            train_idx = np.where(train_labels == cls)[0]
            test_idx = np.where(test_labels == cls)[0]
            rng.shuffle(train_idx)
            rng.shuffle(test_idx)
            train_chunks = np.split(
                train_idx, np.cumsum(allocate_counts(len(train_idx), probabilities[cls]))[:-1]
            )
            test_chunks = np.split(
                test_idx, np.cumsum(allocate_counts(len(test_idx), probabilities[cls]))[:-1]
            )
            for client, (tr, te) in enumerate(zip(train_chunks, test_chunks)):
                nv = (
                    min(max(1, int(round(len(tr) * cfg.validation_fraction))), len(tr) - 1)
                    if len(tr) >= 2
                    else 0
                )
                groups["validation"][client].extend(tr[:nv].tolist())
                groups["train"][client].extend(tr[nv:].tolist())
                groups["test"][client].extend(te.tolist())
        minimums = {
            "train": cfg.min_train_samples,
            "validation": cfg.min_validation_samples,
            "test": cfg.min_test_samples,
        }
        if all((min(map(len, groups[s])) >= minimums[s] for s in groups)):
            arrays = {
                f"{split}_{i}": np.asarray(rng.permutation(ids), dtype=np.int64)
                for split, clients in groups.items()
                for i, ids in enumerate(clients)
            }
            arrays["allocation"] = probabilities
            return (arrays, attempt + 1)
    raise ValueError("Partition minimums were not met; no invalid/empty-client split was returned")


def verify_partitions(arrays, train_labels, test_labels, cfg):
    allocation = arrays["allocation"]
    if (
        allocation.shape != (len(np.unique(train_labels)), cfg.num_clients)
        or not np.isfinite(allocation).all()
        or (allocation < 0).any()
        or (not np.allclose(allocation.sum(axis=1), 1))
    ):
        raise ValueError("Invalid shared allocation probabilities")
    for splits, labels in [(("train", "validation"), train_labels), (("test",), test_labels)]:
        indices = np.concatenate(
            [arrays[f"{s}_{i}"] for s in splits for i in range(cfg.num_clients)]
        )
        if not np.array_equal(np.sort(indices), np.arange(len(labels))):
            raise ValueError("Partitions have missing, duplicate or out-of-range indices")
    for split, minimum in [
        ("train", cfg.min_train_samples),
        ("validation", cfg.min_validation_samples),
        ("test", cfg.min_test_samples),
    ]:
        if min((len(arrays[f"{split}_{i}"]) for i in range(cfg.num_clients))) < minimum:
            raise ValueError("Cached client minimum is invalid")


## 8. Paired loaders and initialization

All methods share partition, classifier-initialization and participation hashes for each dataset, concentration and seed. Summary loaders contain training examples only.

In [ ]:
class PairedTrainingView(Dataset):

    def __init__(self, dataset, indices, seed, client, round_idx, epoch):
        self.dataset, self.indices = (dataset, list(map(int, indices)))
        self.seed, self.client, self.round_idx, self.epoch = (seed, client, round_idx, epoch)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        raw_index = self.indices[index]
        with cpu_random_stream(
            role_seed(self.seed, "augmentation", self.client, self.round_idx, self.epoch, raw_index)
        ):
            return self.dataset[raw_index]


class PairedClientLoader:
    """Private per-client/round/epoch shuffle and augmentation streams."""

    def __init__(self, dataset, indices, cfg, client):
        self.source, self.indices, self.cfg, self.client = (dataset, indices, cfg, client)
        self.dataset = Subset(dataset, indices.tolist())
        self.set_round(0)

    def set_round(self, round_idx):
        self.round_idx, self.epoch = (round_idx, 0)

    def __iter__(self):
        epoch = self.epoch
        self.epoch += 1
        view = PairedTrainingView(
            self.source, self.indices, self.cfg.seed, self.client, self.round_idx, epoch
        )
        generator = torch.Generator().manual_seed(
            role_seed(self.cfg.seed, "minibatches", self.client, self.round_idx, epoch)
        )
        return iter(
            DataLoader(
                view,
                batch_size=self.cfg.batch_size,
                shuffle=True,
                num_workers=self.cfg.num_workers,
                generator=generator,
                pin_memory=torch.cuda.is_available(),
                drop_last=False,
            )
        )

    def __len__(self):
        return (len(self.indices) + self.cfg.batch_size - 1) // self.cfg.batch_size


def deterministic_loaders(dataset, parts, cfg):
    return [
        DataLoader(
            Subset(dataset, ids.tolist()),
            batch_size=cfg.batch_size * 4,
            shuffle=False,
            num_workers=cfg.num_workers,
            generator=torch.Generator().manual_seed(role_seed(cfg.seed, "evaluation-loader", i)),
            pin_memory=torch.cuda.is_available(),
        )
        for i, ids in enumerate(parts)
    ]


def isolated_model_state(factory, cfg, role):
    with cpu_random_stream(role_seed(cfg.seed, role, cfg.dataset)):
        model = factory()
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


class FairRunData:
    """Preserved partition/initialization, with metadata held in memory."""

    def __init__(self, cfg, bundle):
        train, test, tr_y, te_y, channels, size, classes = bundle
        arrays, attempts = make_common_partitions(tr_y, te_y, cfg)
        verify_partitions(arrays, tr_y, te_y, cfg)
        self.arrays = arrays
        self.train_parts, self.validation_parts, self.test_parts = [
            [arrays[f"{s}_{i}"] for i in range(cfg.num_clients)]
            for s in ("train", "validation", "test")
        ]
        self.base_state = isolated_model_state(
            lambda: SimpleCNN(channels, classes), cfg, "classifier-initialization"
        )
        view = copy.copy(train)
        view.transform = test.transform
        self.train_loaders = [
            PairedClientLoader(train, ids, cfg, i) for i, ids in enumerate(self.train_parts)
        ]
        self.assignment_loaders = deterministic_loaders(view, self.train_parts, cfg)
        self.evaluation_loaders = deterministic_loaders(
            test if cfg.evaluation_split == "test" else view,
            self.test_parts if cfg.evaluation_split == "test" else self.validation_parts,
            cfg,
        )
        self.schedule = [common_selection(cfg, r) for r in range(1, cfg.rounds + 1)]
        self.manifest = dict(
            config=asdict(cfg),
            partition_hash=array_hash(arrays),
            partition_attempts=attempts,
            initial_state_hash=array_hash({k: v.numpy() for k, v in self.base_state.items()}),
            schedule_hash=array_hash({"participants": np.asarray(self.schedule)}),
            packages={
                p: importlib.metadata.version(p)
                for p in (
                    "torch",
                    "torchvision",
                    "numpy",
                    "pandas",
                    "scipy",
                    "scikit-learn",
                    "hdbscan",
                )
            },
            device=str(get_device()),
            cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            deterministic_algorithms=True,
            classifier_pooling="equivalent_linear_adaptive_average_v1",
        )

    def set_round(self, round_idx):
        for loader in self.train_loaders:
            loader.set_round(round_idx)


## 9. Classifier

The shared three-block CNN preserves deterministic adaptive-average pooling bins.

In [ ]:
class DeterministicAdaptiveAvgPool2d(nn.AdaptiveAvgPool2d):
    """Same adaptive averaging bins and state_dict as nn.AdaptiveAvgPool2d.

    Avoid its unsupported deterministic CUDA backward: express the fixed linear
    average as F.linear. No trainable weights, RNG calls or model-state changes.
    CUBLAS_WORKSPACE_CONFIG is set in the dependency cell for strict CUDA matmul.
    """

    def forward(self, x):
        if x.ndim not in (3, 4):
            raise ValueError("AdaptiveAvgPool2d expects CHW or NCHW input")
        height, width = x.shape[-2:]
        target = (
            (self.output_size, self.output_size)
            if isinstance(self.output_size, int)
            else self.output_size
        )
        out_h = height if target[0] is None else target[0]
        out_w = width if target[1] is None else target[1]
        if min(height, width, out_h, out_w) < 1:
            raise ValueError("Pooling dimensions must be positive")
        if (height, width) == (out_h, out_w):
            return x

        def bin_weights(input_size, output_size):
            output = torch.arange(output_size, device=x.device)
            start = torch.div(output * input_size, output_size, rounding_mode="floor")
            end = torch.div(
                (output + 1) * input_size + output_size - 1, output_size, rounding_mode="floor"
            )
            positions = torch.arange(input_size, device=x.device)
            mask = (positions[None, :] >= start[:, None]) & (positions[None, :] < end[:, None])
            return mask.to(x.dtype) / (end - start).to(x.dtype)[:, None]

        rows, columns = (bin_weights(height, out_h), bin_weights(width, out_w))
        weights = (rows[:, None, :, None] * columns[None, :, None, :]).reshape(
            out_h * out_w, height * width
        )
        return F.linear(x.flatten(-2), weights).reshape(*x.shape[:-2], out_h, out_w)


class SimpleCNN(nn.Module):

    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            DeterministicAdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


## 10. MDFE gate

The gate combines channel means, peaks and distortion. Disabled MDFE returns its input and freezes gate parameters.

In [ ]:
class MDFEGate(nn.Module):
    """Channel gate combining channel means, peaks and distortion statistics."""

    MODES = {
        "full",
        "off",
        "mean_only",
        "peak_only",
        "mean_peak",
        "fixed_distortion",
        "std_distortion",
    }

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.base = nn.Sequential(
            nn.Linear(channels * 2, hidden), nn.ReLU(inplace=True), nn.Linear(hidden, channels)
        )
        self.rho_raw = nn.Parameter(torch.ones(channels))
        self.rho_mode = "original"
        self.zeta = nn.Parameter(torch.zeros(channels))
        self.gamma = nn.Parameter(torch.tensor(1.0))
        self.mode = "full"

    def configure_rho(self, mode):
        self.rho_mode = mode
        with torch.no_grad():
            self.rho_raw.fill_(float(np.log(np.expm1(1.0))) if mode == "positive" else 1.0)

    def effective_rho(self):
        return F.softplus(self.rho_raw) if self.rho_mode == "positive" else self.rho_raw

    def configure_mode(self, mode):
        if mode not in self.MODES:
            raise ValueError(f"Unknown MDFE mode: {mode}")
        self.mode = mode
        for parameter in self.base.parameters():
            parameter.requires_grad_(mode != "off")
        for parameter in (self.rho_raw, self.zeta, self.gamma):
            parameter.requires_grad_(mode in {"full", "std_distortion"})

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == "off":
            return x
        mean_act = x.mean(dim=(2, 3))
        peak_act = x.amax(dim=(2, 3))
        if self.mode in {"mean_only", "peak_only", "mean_peak"}:
            mean_input = torch.zeros_like(mean_act) if self.mode == "peak_only" else mean_act
            peak_input = torch.zeros_like(peak_act) if self.mode == "mean_only" else peak_act
            gate = torch.sigmoid(self.base(torch.cat([mean_input, peak_input], dim=1)))
        else:
            distortion = peak_act - mean_act
            if self.mode == "std_distortion":
                distortion = torch.sqrt(x.var(dim=(2, 3), unbiased=False) + 1e-08)
            base_gate = self.base(torch.cat([mean_act, peak_act], dim=1))
            if self.mode == "fixed_distortion":
                dist_gate = torch.sigmoid(distortion)
                gamma = F.softplus(x.new_tensor(1.0))
            else:
                dist_gate = torch.sigmoid(distortion * self.effective_rho() + self.zeta)
                gamma = F.softplus(self.gamma)
            gate = torch.sigmoid(base_gate - gamma * dist_gate)
        return x * gate[:, :, None, None]


## 11. Convolutional autoencoder

Two strided convolutions encode images into 128 latent coordinates; the decoder reconstructs the image.

In [ ]:
class MDFEAutoencoder(nn.Module):

    def __init__(self, in_channels: int, image_size: int, latent_dim: int):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.mdfe = MDFEGate(64)
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, image_size, image_size)
            feat = self.mdfe(self.encoder_conv(dummy))
        self.feature_shape = tuple(feat.shape[1:])
        flat_dim = int(np.prod(self.feature_shape))
        self.to_latent = nn.Linear(flat_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, flat_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, in_channels, kernel_size=4, stride=2, padding=1),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.mdfe(self.encoder_conv(x))
        return self.to_latent(torch.flatten(feat, start_dim=1))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        feat = self.from_latent(z).view(z.shape[0], *self.feature_shape)
        return self.decoder(feat)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        recon = self.decode(z)
        return (recon, z)


## 12. Base latent loss and CALD

Every variant retains reconstruction MSE (weight 1.0) and the class-structured term (weight 0.05). Full CALD uses a fixed 64/64 split, 20-round warm-up and 20-round ramp.

In [ ]:
def class_structured_loss(z: torch.Tensor, y: torch.Tensor, margin: float) -> torch.Tensor:
    unique_labels = torch.unique(y)
    if len(unique_labels) == 0:
        return z.new_tensor(0.0)
    prototypes = []
    compactness_terms = []
    for cls in unique_labels:
        mask = y == cls
        z_cls = z[mask]
        proto = z_cls.mean(dim=0)
        prototypes.append(proto)
        compactness_terms.append(((z_cls - proto) ** 2).sum(dim=1).mean())
    compactness = torch.stack(compactness_terms).mean()
    if len(prototypes) < 2:
        return compactness
    proto = torch.stack(prototypes)
    distances = torch.pdist(proto, p=2)
    separation = F.relu(margin - distances).pow(2).mean()
    return compactness + separation


def cald_curriculum(round_idx: int, cfg: ExperimentConfig) -> float:
    if round_idx <= cfg.cald_warmup_rounds:
        return 0.0
    ramp_pos = round_idx - cfg.cald_warmup_rounds
    return min(1.0, ramp_pos / max(1, cfg.cald_ramp_rounds))


class CALDRouter(nn.Module):
    """Shared coordinate allocation; only adaptive mode adds trainable weights."""

    def __init__(self, cfg: ExperimentConfig):
        super().__init__()
        self.mode = cfg.cald_mode
        self.split = cfg.invariant_dim
        self.temperature = cfg.router_temperature
        hard = (torch.arange(cfg.latent_dim) < self.split).float()
        self.register_buffer("hard_gate", hard)
        initial = (2 * hard - 1) * cfg.router_init_logit
        if self.mode == "adaptive":
            self.logits = nn.Parameter(initial)
        else:
            self.register_buffer("logits", initial)

    def gate(self) -> torch.Tensor:
        if self.mode in {"fixed_split", "legacy", "off"}:
            return self.hard_gate
        return torch.sigmoid(self.logits / self.temperature)

    def raw_branches(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if self.mode == "fixed_split":
            return (z[:, : self.split], z[:, self.split :])
        if self.mode == "legacy":
            return torch.chunk(F.normalize(z, dim=1), 2, dim=1)
        g = self.gate()
        return (z * g, z * (1 - g))

    def forward(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        inv, clu = self.raw_branches(z)
        if self.mode == "legacy":
            return (inv, clu)
        return (F.normalize(inv, dim=1, eps=1e-08), F.normalize(clu, dim=1, eps=1e-08))


def cald_loss(
    z: torch.Tensor,
    prev_cluster: int | None,
    prototypes: dict[str, Any],
    round_idx: int,
    cfg: ExperimentConfig,
    router: CALDRouter,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    """Return weighted components whose sum equals the CALD contribution."""
    keys = ("cald_inv", "cald_clu", "cald_margin", "route_soft", "route_budget")
    terms = {key: z.new_zeros(()) for key in keys}
    kappa = cald_curriculum(round_idx, cfg)
    if cfg.cald_mode == "off" or kappa <= 0:
        return (z.new_zeros(()), terms)
    features = z.mean(dim=0, keepdim=True) if cfg.cald_target == "batch_mean" else z
    inv, clu = router(features)
    raw_inv, raw_clu = router.raw_branches(features)
    valid_inv = raw_inv.norm(dim=1) > 1e-08
    valid_clu = raw_clu.norm(dim=1) > 1e-08

    def valid_target(value, branch):
        if value is None:
            return None
        target = torch.as_tensor(value, dtype=z.dtype, device=z.device).detach()
        if target.shape != branch.shape[1:]:
            raise ValueError("Prototype shape does not match CALD mode; start a compatible run.")
        if not torch.isfinite(target).all():
            raise ValueError("Nonfinite CALD prototype.")
        return target if target.norm() > 1e-08 else None

    target = valid_target(prototypes.get("inv"), inv)
    if target is not None and valid_inv.any():
        terms["cald_inv"] = (
            kappa * cfg.lambda_inv * (inv[valid_inv] - target).square().sum(1).mean()
        )
    cluster_protos = prototypes.get("cluster", {})
    own = valid_target(cluster_protos.get(prev_cluster), clu)
    if own is not None and valid_clu.any():
        terms["cald_clu"] = kappa * cfg.lambda_clu * (clu[valid_clu] - own).square().sum(1).mean()
        repulsion = []
        for cluster_id, value in cluster_protos.items():
            if cluster_id == prev_cluster:
                continue
            other = valid_target(value, clu)
            if other is not None:
                distance = (clu[valid_clu] - other).norm(dim=1)
                repulsion.append(F.relu(cfg.margin_clu - distance).square().mean())
        if repulsion:
            terms["cald_margin"] = kappa * cfg.lambda_margin * torch.stack(repulsion).mean()
    if cfg.cald_mode == "adaptive":
        g = router.gate()
        terms["route_soft"] = kappa * cfg.lambda_route_soft * (g * (1 - g)).mean()
        budget = F.relu(cfg.router_min_fraction - g.mean()).square()
        budget = budget + F.relu(g.mean() - cfg.router_max_fraction).square()
        terms["route_budget"] = kappa * cfg.lambda_route_budget * budget
    return (sum(terms.values()), terms)


## 13. Autoencoder construction

The full method constructs a convolutional AE with MDFE and fixed-split CALD. Module variants have a separate notebook.

In [ ]:
def make_autoencoder(in_channels: int, image_size: int, cfg: ExperimentConfig) -> MDFEAutoencoder:
    model = MDFEAutoencoder(in_channels, image_size, cfg.latent_dim)
    model.mdfe.configure_rho(cfg.mdfe_rho_mode)
    model.mdfe.configure_mode(cfg.mdfe_mode)
    model.router = CALDRouter(cfg)
    return model


def mdfe_parameter_diagnostics(ae_state, cfg):
    raw = ae_state["mdfe.rho_raw"]
    rho = F.softplus(raw) if cfg.mdfe_rho_mode == "positive" else raw
    return {
        "mdfe_rho_min": float(rho.min()),
        "mdfe_rho_mean": float(rho.mean()),
        "mdfe_rho_max": float(rho.max()),
        "mdfe_rho_negative_fraction": float((rho < 0).float().mean()),
        "mdfe_zeta_mean": float(ae_state["mdfe.zeta"].mean()),
        "mdfe_gamma_effective": float(F.softplus(ae_state["mdfe.gamma"])),
        "mdfe_distortion_active": int(
            cfg.mdfe_mode in {"full", "fixed_distortion", "std_distortion"}
        ),
    }


## 14. Training-only client summaries

Compute summaries from the deterministic training view after the local update.

In [ ]:
def server_router(ae_state: dict[str, torch.Tensor], cfg: ExperimentConfig) -> CALDRouter:
    router = CALDRouter(cfg)
    router.load_state_dict(
        {
            key.removeprefix("router."): value
            for key, value in ae_state.items()
            if key.startswith("router.")
        }
    )
    return router.eval()


def router_diagnostics(router, summaries):
    g = router.gate().detach().cpu().numpy()
    with torch.no_grad():
        inv, clu = router.raw_branches(torch.as_tensor(summaries, dtype=torch.float32))
    return {
        "gate_mean": float(g.mean()),
        "gate_min": float(g.min()),
        "gate_max": float(g.max()),
        "gate_effective_inv_dims": float(g.sum()),
        "gate_hard_inv_dims": int((g >= 0.5).sum()),
        "gate_saturation": float(((g < 0.05) | (g > 0.95)).mean()),
        "summary_inv_norm": float(inv.norm(dim=1).mean()),
        "summary_clu_norm": float(clu.norm(dim=1).mean()),
        "summary_inv_zero_fraction": float((inv.norm(dim=1) <= 1e-08).float().mean()),
        "summary_clu_zero_fraction": float((clu.norm(dim=1) <= 1e-08).float().mean()),
    }


@torch.no_grad()
def compute_client_summary(autoencoder, loader, device, router=None):
    autoencoder.eval()
    router = copy.deepcopy(router if router is not None else autoencoder.router).to(device).eval()
    totals = {}
    count = 0
    for x, _ in loader:
        z = autoencoder.encode(x.to(device, non_blocking=True))
        inv, clu = router(z)
        for key, value in (("raw", z), ("inv", inv), ("clu", clu)):
            total = value.double().sum(0).cpu().numpy()
            totals[key] = totals.get(key, 0) + total
        count += len(x)
    if count == 0:
        raise ValueError("Empty summary loader")
    return {key: l2_normalize_np((value / count)[None])[0] for key, value in totals.items()}


## 15. Local classifier and autoencoder training

Classifier and AE use separate Adam optimizers on the same minibatches.

In [ ]:
def train_one_client(
    client_id,
    classifier_state,
    ae_state,
    loader,
    cfg,
    in_channels,
    image_size,
    num_classes,
    prev_cluster,
    prototypes,
    device,
    round_idx,
    summary_loader=None,
    summary_router=None,
    train_ae=True,
):
    classifier = SimpleCNN(in_channels, num_classes).to(device)
    classifier.load_state_dict(classifier_state)
    classifier.train()
    autoencoder = None
    if cfg.uses_latentfed:
        autoencoder = make_autoencoder(in_channels, image_size, cfg).to(device)
        if ae_state is not None:
            autoencoder.load_state_dict(ae_state)
        autoencoder.train()
    optimizer = torch.optim.Adam(
        classifier.parameters(), lr=cfg.classifier_lr, weight_decay=cfg.weight_decay
    )
    ae_optimizer = None
    if autoencoder is not None and train_ae:
        groups = [
            {
                "params": [
                    p
                    for name, p in autoencoder.named_parameters()
                    if not name.startswith("router.")
                ],
                "weight_decay": cfg.weight_decay,
            }
        ]
        routing_params = [p for p in autoencoder.router.parameters() if p.requires_grad]
        if routing_params:
            groups.append({"params": routing_params, "weight_decay": 0.0})
        ae_optimizer = torch.optim.Adam(groups, lr=cfg.autoencoder_lr)
    reference = (
        {name: p.detach().clone() for name, p in classifier.named_parameters()}
        if cfg.uses_fedprox
        else None
    )
    totals = {
        key: 0.0
        for key in (
            "loss_ce",
            "loss_prox",
            "loss_rec",
            "loss_structure",
            "loss_cald_inv",
            "loss_cald_clu",
            "loss_cald_margin",
            "loss_route_soft",
            "loss_route_budget",
        )
    }
    total_loss, total_correct, total_seen = (0.0, 0, 0)
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = (x.to(device, non_blocking=True), y.to(device, non_blocking=True))
            optimizer.zero_grad(set_to_none=True)
            if ae_optimizer is not None:
                ae_optimizer.zero_grad(set_to_none=True)
            logits = classifier(x)
            losses = {key: x.new_zeros(()) for key in totals}
            losses["loss_ce"] = F.cross_entropy(logits, y)
            if reference is not None:
                losses["loss_prox"] = (
                    0.5
                    * cfg.fedprox_mu
                    * sum(
                        (
                            (p - reference[name]).square().sum()
                            for name, p in classifier.named_parameters()
                        )
                    )
                )
            if autoencoder is not None and train_ae:
                recon, z = autoencoder(x)
                losses["loss_rec"] = cfg.lambda_rec * F.mse_loss(recon, x)
                losses["loss_structure"] = cfg.lambda_str * class_structured_loss(
                    z, y, cfg.margin_cls
                )
                _, cald_terms = cald_loss(
                    z, prev_cluster, prototypes, round_idx, cfg, autoencoder.router
                )
                losses.update({"loss_" + key: value for key, value in cald_terms.items()})
            loss = sum(losses.values())
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Nonfinite training loss: client={client_id}, round={round_idx}"
                )
            loss.backward()
            optimizer.step()
            if ae_optimizer is not None:
                ae_optimizer.step()
            size = y.numel()
            total_seen += size
            total_correct += int((logits.argmax(1) == y).sum().item())
            total_loss += float(loss.detach()) * size
            for key, value in losses.items():
                totals[key] += float(value.detach()) * size
    if total_seen == 0:
        raise ValueError(f"Empty training loader for client {client_id}")
    summary = (
        compute_client_summary(
            autoencoder,
            summary_loader if summary_loader is not None else loader,
            device,
            summary_router,
        )
        if autoencoder is not None
        else None
    )
    return {
        "client_id": client_id,
        "classifier_state": state_dict_to_cpu(classifier.state_dict()),
        "ae_state": (
            state_dict_to_cpu(autoencoder.state_dict()) if autoencoder is not None else None
        ),
        "summary": summary,
        "num_train": len(loader.dataset),
        "num_seen": total_seen,
        "train_loss": total_loss / total_seen,
        "train_acc": total_correct / total_seen,
    }


def warmup_one_client(ae_state, loader, cfg, channels, size, device):
    ae = make_autoencoder(channels, size, cfg).to(device)
    ae.load_state_dict(ae_state)
    ae.train()
    optimizer = torch.optim.Adam(
        [
            p
            for name, p in ae.named_parameters()
            if p.requires_grad and (not name.startswith("router."))
        ],
        lr=cfg.autoencoder_lr,
        weight_decay=cfg.weight_decay,
    )
    count, total = (0, 0.0)
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = (x.to(device), y.to(device))
            optimizer.zero_grad(set_to_none=True)
            recon, z = ae(x)
            loss = cfg.lambda_rec * F.mse_loss(recon, x) + cfg.lambda_str * class_structured_loss(
                z, y, cfg.margin_cls
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite warm-up loss")
            loss.backward()
            optimizer.step()
            count += len(y)
            total += float(loss.detach()) * len(y)
    return dict(
        ae_state=state_dict_to_cpu(ae.state_dict()),
        num_train=len(loader.dataset),
        num_seen=count,
        loss=total / count,
    )


## 16. Server geometry and clustering

Normalize latent summaries, cluster with HDBSCAN, and align cluster identities between rounds.

In [ ]:
def route_summaries(summary_matrix, router, cfg):
    with torch.no_grad():
        inv, clu = router(torch.as_tensor(summary_matrix, dtype=torch.float32))
    inv, clu = (inv.numpy().astype(np.float64), clu.numpy().astype(np.float64))
    full = np.concatenate([inv, clu], axis=1) / np.sqrt(2.0)
    if cfg.cald_mode in {"off", "legacy"}:
        full = summary_matrix.copy()
    return (full, inv, clu)


def make_geometry(summaries, router, cfg, client_ids, cache, round_idx):
    raw = np.stack([s["raw"] for s in summaries])
    if cfg.dp_noise_std:
        raw = l2_normalize_np(
            raw
            + np.random.default_rng(role_seed(cfg.seed, "summary-noise", round_idx)).normal(
                0, cfg.dp_noise_std, raw.shape
            )
        )
    if cfg.cluster_geometry == "full":
        geometry, inv, clu = route_summaries(raw, router, cfg)
    else:
        inv, clu = (np.stack([s[key] for s in summaries]) for key in ("inv", "clu"))
        geometry = clu.copy()
        if cfg.dp_noise_std:
            geometry = l2_normalize_np(
                geometry
                + np.random.default_rng(role_seed(cfg.seed, "cluster-noise", round_idx)).normal(
                    0, cfg.dp_noise_std, geometry.shape
                )
            )
    for j, client in enumerate(client_ids):
        if cfg.summary_ema_beta and client in cache:
            geometry[j] = unit_ema(cache[client], geometry[j], cfg.summary_ema_beta)
        cache[client] = geometry[j].copy()
    if not all((np.isfinite(x).all() for x in (geometry, inv, clu))):
        raise FloatingPointError("Nonfinite latent summaries")
    return (geometry, inv, clu, raw)


def cluster_summaries(geometry, cfg):
    if len(geometry) < max(2, cfg.cluster_min_size):
        return np.full(len(geometry), -1, dtype=int)
    if hdbscan is None:
        raise RuntimeError("HDBSCAN is required; no substitute backend")
    return (
        hdbscan.HDBSCAN(
            min_cluster_size=cfg.cluster_min_size,
            min_samples=cfg.cluster_min_samples,
            metric="euclidean",
            prediction_data=False,
            core_dist_n_jobs=1,
            approx_min_span_tree=True,
        )
        .fit_predict(geometry)
        .astype(int)
    )


def normalized_centroid(rows):
    rows = np.asarray(rows)
    if not len(rows):
        return None
    valid = rows[np.linalg.norm(rows, axis=1) > 1e-08]
    if not len(valid):
        return None
    value = valid.mean(0)
    return l2_normalize_np(value[None])[0] if np.linalg.norm(value) > 1e-08 else None


def unit_ema(previous, current, beta):
    value = beta * previous + (1 - beta) * current
    return l2_normalize_np(value[None])[0] if np.linalg.norm(value) > 1e-08 else current.copy()


def align_clusters(raw_labels, geometry, previous_prototypes, next_id, cfg):
    raw_ids = sorted((int(k) for k in np.unique(raw_labels) if k >= 0))
    centroids = {k: normalized_centroid(geometry[raw_labels == k]) for k in raw_ids}
    old_ids = sorted(
        (k for k, p in previous_prototypes.items() if p is not None and np.linalg.norm(p) > 1e-08)
    )
    aligned = np.full(len(raw_labels), -1, dtype=int)
    remap, matched = ({}, set())
    valid_ids = [k for k in raw_ids if centroids[k] is not None]
    if valid_ids and old_ids:
        cost = 1 - cosine_similarity(
            np.stack([centroids[k] for k in valid_ids]),
            np.stack([previous_prototypes[k] for k in old_ids]),
        )
        threshold = cfg.alignment_max_cosine_distance
        if cfg.cluster_alignment == "hungarian":
            padded = np.concatenate(
                [
                    np.where(cost <= threshold, cost, 1000000.0),
                    np.full((len(valid_ids), len(valid_ids)), threshold + 1e-06),
                ],
                axis=1,
            )
            rows, cols = linear_sum_assignment(padded)
            pairs = [
                (i, j) for i, j in zip(rows, cols) if j < len(old_ids) and cost[i, j] <= threshold
            ]
        else:
            pairs, used = ([], set())
            for i in range(len(valid_ids)):
                available = [j for j in range(len(old_ids)) if j not in used]
                if available:
                    j = min(available, key=lambda j: cost[i, j])
                    if cost[i, j] <= threshold:
                        pairs.append((i, j))
                        used.add(j)
        for i, j in pairs:
            remap[valid_ids[i]] = old_ids[j]
            matched.add(old_ids[j])
    for k in raw_ids:
        if k not in remap:
            remap[k] = next_id
            next_id += 1
        aligned[raw_labels == k] = remap[k]
    current = {remap[k]: centroids[k] for k in raw_ids if centroids[k] is not None}
    return (aligned, current, matched, next_id)


## 17. Routing and prototypes

Route noise clients, exclude them from prototype estimation, and aggregate classifier states within assigned clusters.

In [ ]:
def route_outliers(
    aligned, geometry, current_centroids, previous_valid, previous_routes, existing_models
):
    routed = aligned.copy()
    for i in np.flatnonzero(aligned < 0):
        if previous_valid[i] in current_centroids:
            routed[i] = previous_valid[i]
        elif current_centroids:
            ids = sorted(current_centroids)
            sims = cosine_similarity(
                geometry[i : i + 1], np.stack([current_centroids[k] for k in ids])
            )[0]
            routed[i] = ids[int(np.argmax(sims))]
        else:
            routed[i] = (
                previous_routes[i]
                if previous_routes[i] in existing_models
                else min(existing_models)
            )
    assert (routed >= 0).all()
    return routed


def update_prototypes(geometry, inv, clu, aligned, routed, raw_labels, previous, matched, cfg):
    eligible = (
        raw_labels >= 0 if cfg.exclude_noise_from_prototypes else np.ones(len(raw_labels), bool)
    )
    labels = aligned if cfg.exclude_noise_from_prototypes else routed
    if not eligible.any():
        return copy.deepcopy(previous)
    output = {"inv": None, "cluster": {}, "full_cluster": {}}
    current_inv = normalized_centroid(inv[eligible])
    old_inv = previous.get("inv")
    output["inv"] = (
        unit_ema(old_inv, current_inv, cfg.prototype_ema_beta)
        if current_inv is not None and old_inv is not None
        else current_inv
    )
    for k in sorted((int(k) for k in np.unique(labels[eligible]) if k >= 0)):
        for key, features in (("cluster", clu), ("full_cluster", geometry)):
            value = normalized_centroid(features[eligible & (labels == k)])
            if value is None:
                continue
            old = previous.get(key, {}).get(k)
            output[key][k] = (
                unit_ema(old, value, cfg.prototype_ema_beta)
                if k in matched and old is not None
                else value
            )
    return output


def aggregate_client_models(client_results, assignments):
    groups = {}
    for result in client_results:
        groups.setdefault(int(assignments[result["client_id"]]), []).append(result)
    return {
        k: weighted_average_states(
            [r["classifier_state"] for r in rows], [r["num_train"] for r in rows]
        )
        for k, rows in groups.items()
    }


## 18. Test accuracy

W-Acc pools correct/total counts; M-Acc averages client accuracies. MC/BC/WC summarize pooled cluster accuracies.

In [ ]:
@torch.no_grad()
def evaluate_model(
    model: nn.Module, loader: DataLoader, device: torch.device
) -> tuple[int, int, float]:
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y, reduction="sum")
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.numel())
        loss_sum += float(loss.item())
    return (correct, total, loss_sum / max(1, total))


def evaluate_table_iii_metrics(
    cluster_model_states: dict[int, dict[str, torch.Tensor]],
    client_assignments: np.ndarray,
    client_test_loaders: list[DataLoader],
    cfg: ExperimentConfig,
    in_channels: int,
    num_classes: int,
    device: torch.device,
    return_client_rows: bool = False,
    personalized_states=None,
):
    client_rows = []
    cluster_totals: dict[int, dict[str, float]] = {}
    for client_id, loader in enumerate(client_test_loaders):
        cluster_id = int(client_assignments[client_id])
        if cluster_id not in cluster_model_states:
            raise ValueError("Client routed to missing model")
        model = SimpleCNN(in_channels, num_classes).to(device)
        model.load_state_dict(
            personalized_states[client_id]
            if personalized_states is not None
            else cluster_model_states[cluster_id]
        )
        correct, total, loss = evaluate_model(model, loader, device)
        del model
        if total == 0:
            raise ValueError(f"Client {client_id} has no evaluation examples; repair the partition")
        acc = correct / total
        client_rows.append(
            {
                "client_id": client_id,
                "cluster_id": cluster_id,
                "correct": correct,
                "total": total,
                "acc": acc,
                "loss": loss,
                "evaluation_split": cfg.evaluation_split,
            }
        )
        cluster_totals.setdefault(cluster_id, {"correct": 0, "total": 0})
        cluster_totals[cluster_id]["correct"] += correct
        cluster_totals[cluster_id]["total"] += total
    if device.type == "cuda":
        torch.cuda.empty_cache()
    client_df = pd.DataFrame(client_rows)
    weighted_acc = client_df["correct"].sum() / max(1, client_df["total"].sum())
    macro_acc = float(client_df["acc"].mean())
    cluster_accs = [v["correct"] / max(1, v["total"]) for v in cluster_totals.values()]
    unique_clusters = np.unique(client_assignments)
    result = {
        "K": int(len(unique_clusters)),
        "Worst-client-Acc": float(client_df["acc"].min()),
        "P10-client-Acc": float(client_df["acc"].quantile(0.1)),
        "W-Acc": float(weighted_acc),
        "M-Acc": float(macro_acc),
        "MC": float(np.mean(cluster_accs)) if cluster_accs else float("nan"),
        "BC": float(np.max(cluster_accs)) if cluster_accs else float("nan"),
        "WC": float(np.min(cluster_accs)) if cluster_accs else float("nan"),
    }
    return (result, client_rows) if return_client_rows else result


## 19. Clustering metrics and seed endpoints

Undefined geometric scores remain missing. Endpoints average each metric across the final ten rounds.

In [ ]:
def clustering_metrics(features, labels):
    labels = np.asarray(labels)
    mask = labels >= 0
    k, ni = (len(np.unique(labels[mask])), int(mask.sum()))
    result = dict(
        K=k,
        n_inliers=ni,
        n_outliers=int((~mask).sum()),
        noise_fraction=float((~mask).mean()) if len(mask) else float("nan"),
    )
    result.update(
        {
            key: float("nan")
            for key in ("Silhouette", "Calinski-Harabasz", "Davies-Bouldin", "SSE", "SSE/Inlier")
        }
    )
    if features is None:
        return result
    x = np.asarray(features, dtype=np.float64)
    if x.ndim != 2 or len(x) != len(labels) or (not np.isfinite(x).all()):
        raise ValueError("Invalid clustering metric inputs")
    x, y = (x[mask], labels[mask])
    if k < 2:
        return result
    sse = sum((float(((x[y == c] - x[y == c].mean(0)) ** 2).sum()) for c in np.unique(y)))
    result.update(SSE=sse, **{"SSE/Inlier": sse / ni})
    if k < ni and sse > 0.0:
        result["Calinski-Harabasz"] = float(calinski_harabasz_score(x, y))
    if k < ni and (not np.all(np.linalg.norm(x - x[:1], axis=1) <= 1e-12)):
        result["Silhouette"] = float(silhouette_score(x, y))
        centers = np.stack([x[y == c].mean(0) for c in np.unique(y)])
        if (
            np.min(
                np.linalg.norm(centers[:, None] - centers[None, :], axis=-1)
                + np.eye(k) * 1000000000.0
            )
            > 1e-12
        ):
            result["Davies-Bouldin"] = float(davies_bouldin_score(x, y))
    return result


def temporal_metrics(previous, current, selected, previous_selected):
    overlap = np.intersect1d(selected, previous_selected).astype(int)
    ids = overlap[(previous[overlap] >= 0) & (current[overlap] >= 0)]
    valid = len(ids) >= 2 and min(len(np.unique(previous[ids])), len(np.unique(current[ids]))) >= 2
    return {
        "Temporal ARI": (
            float(adjusted_rand_score(previous[ids], current[ids])) if valid else float("nan")
        ),
        "Temporal ARI with noise": (
            float(adjusted_rand_score(previous[overlap], current[overlap]))
            if len(overlap) >= 2
            else float("nan")
        ),
        "noise_transition_fraction": (
            float(np.mean((previous[overlap] < 0) != (current[overlap] < 0)))
            if len(overlap)
            else float("nan")
        ),
        "temporal_observed_overlap": int(len(overlap)),
        "temporal_inlier_overlap": int(len(ids)),
        "cluster_id_switch_fraction": (
            float(np.mean(previous[ids] != current[ids])) if len(ids) else float("nan")
        ),
    }


CLASS_METRICS = ["W-Acc", "M-Acc", "MC", "BC", "WC"]

CLUSTER_METRICS = [
    "Silhouette",
    "Calinski-Harabasz",
    "Davies-Bouldin",
    "SSE",
    "SSE/Inlier",
    "K",
    "noise_fraction",
    "Temporal ARI",
]

DIAGNOSTIC_METRICS = [
    "n_inliers",
    "n_outliers",
    "K_models",
    "cluster_id_switch_fraction",
    "temporal_inlier_overlap",
    "Temporal ARI with noise",
    "noise_transition_fraction",
    "mdfe_rho_min",
    "mdfe_rho_mean",
    "mdfe_rho_max",
    "mdfe_rho_negative_fraction",
]


def seed_summary(history, cfg):
    frame = pd.DataFrame(history)
    if len(frame) != 100 or frame["round"].tolist() != list(range(1, 101)):
        raise ValueError(
            "A seed score requires exactly rounds 1–100, without duplicates or omissions"
        )
    tail = frame.loc[frame["round"].between(91, 100)]
    out = dict(
        Dataset=cfg.dataset,
        Distribution="IID" if cfg.alpha is None else f"alpha={cfg.alpha}",
        Method=cfg.method,
        Seed=cfg.seed,
        Evaluation=cfg.evaluation_split,
        config_id=cfg.config_id,
        window_start=91,
        window_end=100,
    )
    for metric in CLASS_METRICS + CLUSTER_METRICS + DIAGNOSTIC_METRICS:
        values = tail[metric].to_numpy(dtype=float) if metric in tail else np.full(10, np.nan)
        if np.isinf(values).any():
            raise ValueError(f"Infinite metric {metric}")
        count = int(np.isfinite(values).sum())
        if metric in CLASS_METRICS and count != 10:
            raise ValueError(f"Missing classification observations: {metric}")
        out[metric] = float(np.nanmean(values)) if count else float("nan")
        out[metric + "_valid_rounds"] = count
    out["multicluster_fraction"] = float((tail["K"] >= 2).mean())
    out["classifier_examples"] = int(frame["train_examples"].sum())
    out["ae_examples_normal"] = int(frame["ae_examples"].sum())
    return out


## 20. Experiment grid and diagnostic data

Paper settings are fixed in advance. Synthetic images are used only by smoke mode, with seed 7 and two rounds.

In [ ]:
def make_config(
    dataset="cifar10",
    alpha=0.3,
    seed=42,
    variant="full",
    method="latentfed",
    data_root="./data",
    smoke=False,
):
    if (
        dataset not in PAPER_DATASETS
        or alpha not in PAPER_ALPHAS
        or variant not in SUPPORTED_VARIANTS
    ):
        raise ValueError("Use the documented two datasets, three alpha values and four variants.")
    cfg = ExperimentConfig(
        dataset=dataset,
        alpha=alpha,
        seed=7 if smoke else seed,
        variant=variant,
        method="latentfed_fedavg" if method == "latentfed" else method,
        data_root=str(data_root),
        rounds=2 if smoke else 100,
        num_workers=0 if smoke else 2,
        allow_custom_protocol=smoke,
        min_train_samples=1 if smoke else 10,
        min_validation_samples=1,
        min_test_samples=1,
    )
    if smoke:
        cfg.cald_warmup_rounds = 0
        cfg.cald_ramp_rounds = 1
    if variant in {"no_mdfe", "ae_only"}:
        cfg.mdfe_mode = "off"
    if variant in {"no_cald", "ae_only"}:
        cfg.cald_mode = "off"
        cfg.lambda_inv = cfg.lambda_clu = cfg.lambda_margin = 0.0
        cfg.lambda_route_soft = cfg.lambda_route_budget = 0.0
    if variant == "ae_only":
        cfg.ae_architecture = "fully_connected"
    cfg.validate()
    return cfg


def experiment_plan(
    datasets=PAPER_DATASETS, alphas=PAPER_ALPHAS, seeds=PAPER_SEEDS, variants=("full",), smoke=False
):
    if not datasets or not alphas or (not seeds) or (not variants):
        raise ValueError("The grid may not be empty.")
    if any((x not in PAPER_DATASETS for x in datasets)) or any(
        (x not in PAPER_ALPHAS for x in alphas)
    ):
        raise ValueError("Grid contains an unsupported dataset or alpha.")
    if any((x not in PAPER_SEEDS for x in seeds)) or any(
        (x not in SUPPORTED_VARIANTS for x in variants)
    ):
        raise ValueError("Grid contains an unsupported official seed or variant.")
    selected_alphas = (alphas[0],) if smoke else alphas
    selected_seeds = (7,) if smoke else seeds
    return [
        dict(dataset=d, alpha=a, seed=s, variant=v)
        for d in datasets
        for a in selected_alphas
        for s in selected_seeds
        for v in variants
    ]


class SyntheticImages(Dataset):
    """Small deterministic tensors used only by the explicitly labelled smoke mode."""

    def __init__(self, count, channels, size, classes, seed):
        generator = torch.Generator().manual_seed(seed)
        self.images = torch.randn(count, channels, size, size, generator=generator)
        self.targets = np.arange(count) % classes
        self.transform = None

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return (self.images[index], int(self.targets[index]))


def synthetic_bundle(dataset):
    channels, size, classes = (3, 32, 10) if dataset == "cifar10" else (1, 28, 47)
    train = SyntheticImages(400, channels, size, classes, 1739)
    test = SyntheticImages(200, channels, size, classes, 2718)
    return (train, test, train.targets, test.targets, channels, size, classes)


## 21. Checkpoint serialization and code identity

Record live definitions and RNG/settings locally. Code identity is independent of cell filenames and execution numbers.

In [ ]:
def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(x) for x in value]
    if isinstance(value, np.ndarray):
        return clean(value.tolist())
    if isinstance(value, np.generic):
        return clean(value.item())
    if isinstance(value, float) and (not math.isfinite(value)):
        return None
    if isinstance(value, Path):
        return str(value)
    return value


def json_write(path, value):
    Path(path).write_text(json.dumps(clean(value), indent=2, sort_keys=True, allow_nan=False))


def atomic_checkpoint(path, state):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    os.replace(temporary, path)


def load_checkpoint(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def _code_signature(code):
    def constant(value):
        if isinstance(value, types.CodeType):
            return _code_signature(value)
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if isinstance(value, (tuple, list)):
            return [constant(v) for v in value]
        if isinstance(value, frozenset):
            return sorted([constant(v) for v in value], key=repr)
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)

    return dict(
        bytecode=code.co_code.hex(),
        constants=[constant(v) for v in code.co_consts],
        names=code.co_names,
        variables=code.co_varnames,
        freevars=code.co_freevars,
        cellvars=code.co_cellvars,
        arguments=code.co_argcount,
        keyword_only=code.co_kwonlyargcount,
        positional_only=code.co_posonlyargcount,
        exception_table=code.co_exceptiontable.hex(),
        flags=code.co_flags,
    )


def code_digest():
    """Hash live notebook definitions independently of cell filenames/numbers."""
    definitions = {}
    for name, value in sorted(list(globals().items())):
        if inspect.isfunction(value) and value.__module__ == __name__:
            function = inspect.unwrap(value)
            definitions[name] = dict(
                code=_code_signature(function.__code__),
                defaults=clean(function.__defaults__),
                keyword_defaults=clean(function.__kwdefaults__),
            )
        elif inspect.isclass(value) and value.__module__ == __name__:
            methods = {}
            for key, member in value.__dict__.items():
                if isinstance(member, property):
                    member = member.fget
                elif isinstance(member, (classmethod, staticmethod)):
                    member = member.__func__
                if inspect.isfunction(member):
                    methods[key] = _code_signature(inspect.unwrap(member).__code__)
            definitions[name] = methods
    return hashlib.sha256(
        json.dumps(definitions, sort_keys=True, allow_nan=False).encode()
    ).hexdigest()


## 22. Metrics, five-seed aggregation and ZIP export

Endpoints average rounds 91–100; mean/sample SD use five seed endpoints. ZIPs exclude raw data.

In [ ]:
def write_result(
    folder, cfg, history, manifest, method, smoke=False, complete=False, summary=None, extra=None
):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    from dataclasses import asdict

    json_write(folder / "config.json", asdict(cfg))
    clients = []
    diagnostics = []
    flat = []
    for row in history:
        flat.append(
            {k: v for k, v in row.items() if not isinstance(v, (dict, list, tuple, np.ndarray))}
        )
        for client in row.get("client_metrics", []):
            clients.append(dict(round=row["round"], **client))
        diagnostics.append(
            {k: v for k, v in row.items() if k not in flat[-1] and k != "client_metrics"}
        )
    pd.DataFrame(flat).to_csv(folder / "round_metrics.csv", index=False)
    pd.DataFrame(clients).to_csv(folder / "client_metrics.csv", index=False)
    with (folder / "round_diagnostics.jsonl").open("w") as f:
        for row, item in zip(history, diagnostics):
            f.write(json.dumps(clean(dict(round=row["round"], **item)), allow_nan=False) + "\n")
    json_write(
        folder / "manifest.json",
        dict(
            method=method,
            variant=cfg.variant,
            dataset=cfg.dataset,
            alpha=cfg.alpha,
            seed=cfg.seed,
            mode="smoke" if smoke else "paper",
            complete=complete,
            completed_rounds=len(history),
            publication_eligible=bool(complete and (not smoke) and (len(history) == 100)),
            code_sha256=code_digest(),
            protocol="matched_latentfed_v1",
            **manifest,
            **extra or {}
        ),
    )
    if summary is not None:
        json_write(folder / "seed_summary.json", summary)
    return folder


def export_zip(output, include_checkpoints=False):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    archive = output / "results.zip"
    temporary = output / "results.zip.tmp"
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(output.rglob("*")):
            if not p.is_file() or p in {archive, temporary}:
                continue
            if p.suffix not in {".json", ".csv", ".jsonl"} and (
                not (include_checkpoints and p.suffix == ".pt")
            ):
                continue
            z.write(p, str(p.relative_to(output)))
    temporary.replace(archive)
    return archive


def aggregate_runs(output):
    """Five-seed means/sample SD; never pools rounds or admits smoke results."""
    output = Path(output)
    groups = {}
    signatures = {}
    for path in output.rglob("seed_summary.json"):
        manifest = json.loads((path.parent / "manifest.json").read_text())
        if not manifest["publication_eligible"]:
            continue
        score = json.loads(path.read_text())
        if score["window_start"] != 91 or score["window_end"] != 100:
            raise ValueError("Wrong final window.")
        key = (manifest["method"], manifest["variant"], manifest["dataset"], manifest["alpha"])
        cfg = json.loads((path.parent / "config.json").read_text())
        scientific = {
            k: v
            for k, v in cfg.items()
            if k not in {"seed", "data_root", "num_workers", "checkpoint_every"}
        }
        signature = json.dumps(
            dict(
                configuration=scientific,
                code=manifest["code_sha256"],
                protocol=manifest["protocol"],
            ),
            sort_keys=True,
        )
        if key in signatures and signatures[key] != signature:
            raise ValueError("Seed aggregation would mix configurations or source versions.")
        signatures[key] = signature
        group = groups.setdefault(key, {})
        if manifest["seed"] in group:
            raise ValueError("Duplicate/conflicting seed in aggregate.")
        group[manifest["seed"]] = score
    summary = []
    coverage = []
    required = {42, 101, 2024, 3407, 8888}
    metrics = [
        "W-Acc",
        "M-Acc",
        "MC",
        "BC",
        "WC",
        "Silhouette",
        "Calinski-Harabasz",
        "Davies-Bouldin",
        "SSE",
        "Temporal ARI",
    ]
    for key, group in sorted(groups.items()):
        base = dict(zip(["method", "variant", "dataset", "alpha"], key))
        coverage.append(
            dict(
                **base,
                n_seeds=len(group),
                seeds=";".join(map(str, sorted(group))),
                complete_five_seed_group=set(group) == required
            )
        )
        if set(group) != required:
            continue
        row = dict(**base, n_seeds=5, std_ddof=1, window_start=91, window_end=100)
        for metric in metrics:
            vals = [group[s].get(metric) for s in sorted(required)]
            valid = all(
                (
                    v is not None
                    and math.isfinite(v)
                    and (group[s].get(metric + "_valid_rounds", 10) == 10)
                    for v, s in zip(vals, sorted(required))
                )
            )
            scale = 100 if metric in metrics[:5] else 1
            row[metric + "_mean"] = float(np.mean(vals)) * scale if valid else None
            row[metric + "_std"] = float(np.std(vals, ddof=1)) * scale if valid else None
        summary.append(row)
    pd.DataFrame(summary).to_csv(output / "five_seed_summary.csv", index=False)
    pd.DataFrame(coverage).to_csv(output / "seed_coverage.csv", index=False)
    return (pd.DataFrame(summary), pd.DataFrame(coverage))


## 23. Run identity and final-window scoring

Resume requires matching scientific settings and definitions. Publication endpoints require the complete 100-round history.

In [ ]:
def _run_folder(output, method, cfg, smoke):
    folder = (
        Path(output)
        / ("smoke" if smoke else "paper")
        / method
        / cfg.variant
        / cfg.dataset
        / f"alpha_{cfg.alpha:g}"
        / f"seed_{cfg.seed}_{cfg.config_id}"
    )
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def _state_or_fresh(path, cfg, method, resume):
    if path.exists():
        if not resume:
            raise FileExistsError("Use RESUME=True or choose a fresh output directory.")
        saved = load_checkpoint(path)
        if saved.get("public_code_sha256") != code_digest():
            raise ValueError(
                "Checkpoint definitions differ. Use matching code or a fresh output directory."
            )
        if (
            saved.get("public_method") != method
            or ExperimentConfig(**saved["public_config"]).config_id != cfg.config_id
        ):
            raise ValueError("Checkpoint method or scientific settings differ.")
        return saved
    return dict(public_code_sha256=code_digest(), public_config=asdict(cfg), public_method=method)


def _score(history, cfg, method):
    if len(history) != 100 or [r["round"] for r in history] != list(range(1, 101)):
        raise ValueError("An endpoint requires all 100 ordered rounds.")
    score = dict(
        method=method,
        variant=cfg.variant,
        dataset=cfg.dataset,
        alpha=cfg.alpha,
        seed=cfg.seed,
        window_start=91,
        window_end=100,
    )
    metrics = [
        "W-Acc",
        "M-Acc",
        "MC",
        "BC",
        "WC",
        "Silhouette",
        "Calinski-Harabasz",
        "Davies-Bouldin",
        "SSE",
        "Temporal ARI",
    ]
    for metric in metrics:
        finite = [
            float(r[metric])
            for r in history[-10:]
            if r.get(metric) is not None and math.isfinite(float(r[metric]))
        ]
        score[metric] = sum(finite) / 10 if len(finite) == 10 else None
        score[metric + "_valid_rounds"] = len(finite)
        if metric in {"W-Acc", "M-Acc"} and len(finite) != 10:
            raise ValueError("Test accuracy is missing in the final window.")
    return score


def capture_rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


## 24. Frozen protocol and checkpoint state

Each final-test run is checked against the declared configuration.

In [ ]:
def fresh_suite():
    return dict(
        version=METHOD_IMPLEMENTATION_VERSION,
        runs={},
        active=None,
        decisions=[],
        frozen=None,
        development_complete=False,
    )


def config_digest(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, allow_nan=False).encode()).hexdigest()


def validate_run_phase(cfg, phase, suite):
    cfg.validate()
    if phase == "smoke":
        if not cfg.allow_custom_protocol or cfg.seed not in DEV_SEEDS:
            raise ValueError("Smoke requires a separate development seed and custom short budget")
        return
    if cfg.rounds != 100 or cfg.allow_custom_protocol:
        raise ValueError("Research runs require 100 rounds and the unchanged classifier protocol")
    if phase == "development":
        if cfg.seed not in DEV_SEEDS or cfg.evaluation_split != "validation":
            raise ValueError("Development may use only DEV_SEEDS and validation")
        if suite.get("frozen") is not None:
            raise ValueError("Configuration already frozen: development is closed")
    elif phase == "final":
        frozen = suite.get("frozen")
        if cfg.seed not in FINAL_SEEDS or cfg.evaluation_split != "test" or frozen is None:
            raise ValueError("Final test runs require official seeds and a frozen configuration")
        payload = {k: v for k, v in frozen.items() if k != "digest"}
        if config_digest(payload) != frozen["digest"]:
            raise ValueError("Frozen configuration was altered")
        expected = replace(
            ExperimentConfig(**frozen["config"]),
            dataset=cfg.dataset,
            alpha=cfg.alpha,
            seed=cfg.seed,
            evaluation_split="test",
        )
        if (
            cfg.config_id != expected.config_id
            or cfg.dataset not in frozen["datasets"]
            or cfg.alpha not in frozen["alphas"]
        ):
            raise ValueError("Final run differs from frozen settings")
    else:
        raise ValueError("Unknown experiment phase")


## 25. LatentFed federated loop

The round-by-round method is explicit: local updates → summaries → clustering/routing/prototypes → cluster aggregation → evaluation. The checkpoint path is an argument.

In [ ]:
def run_experiment(
    cfg, phase="development", suite=None, dataset_bundle=None, stop_after=None, checkpoint_path=None
):
    suite = fresh_suite() if suite is None else suite
    validate_run_phase(cfg, phase, suite)
    key = phase + ":" + cfg.config_id
    if key in suite["runs"]:
        return suite["runs"][key]
    if phase != "smoke" and dataset_bundle is not None:
        raise ValueError("Synthetic/custom datasets are permitted only for smoke tests")
    set_seed(cfg.seed)
    device = get_device()
    bundle = (
        dataset_bundle
        if dataset_bundle is not None
        else get_dataset_bundle(cfg.dataset, cfg.data_root)
    )
    _, _, _, _, channels, size, classes = bundle
    fair = FairRunData(cfg, bundle)
    active = suite.get("active")
    if active is not None:
        if active["key"] != key:
            raise ValueError("Resume the active experiment before starting another")
        for identity in (
            "partition_hash",
            "initial_state_hash",
            "schedule_hash",
            "packages",
            "device",
        ):
            if active["manifest"][identity] != fair.manifest[identity]:
                raise ValueError(f"Resume identity mismatch: {identity}")
        state = copy.deepcopy(active)
        restore_rng_state(state["rng"])
    else:
        state = dict(
            key=key,
            phase=phase,
            config=asdict(cfg),
            manifest=fair.manifest,
            round=0,
            warmup_done=0,
            warmup_history=[],
            history=[],
            cluster_models={0: copy.deepcopy(fair.base_state)},
            assignments=np.zeros(cfg.num_clients, dtype=int),
            previous_valid=np.full(cfg.num_clients, -1, dtype=int),
            previous_selected=[],
            prototypes={},
            next_cluster_id=1,
            summary_cache={},
            ae_state=(
                isolated_model_state(
                    lambda: make_autoencoder(channels, size, cfg), cfg, "autoencoder-initialization"
                )
                if cfg.uses_latentfed
                else None
            ),
        )

    def checkpoint():
        state["rng"] = capture_rng_state()
        suite["active"] = copy.deepcopy(state)
        if phase != "smoke":
            save_suite_checkpoint(suite, checkpoint_path)

    started = time.monotonic()
    warmup = cfg.ae_pretrain_rounds if cfg.uses_latentfed else 0
    for w in range(state["warmup_done"], warmup):
        budget_round = cfg.rounds - warmup + w + 1
        clients = fair.schedule[budget_round - 1]
        fair.set_round(-w - 1)
        rows = []
        for client in clients:
            set_seed(role_seed(cfg.seed, "ae-warmup", w, client))
            rows.append(
                warmup_one_client(
                    state["ae_state"], fair.train_loaders[client], cfg, channels, size, device
                )
            )
        state["ae_state"] = weighted_average_states(
            [r["ae_state"] for r in rows], [r["num_train"] for r in rows]
        )
        state["warmup_history"].append(
            dict(
                warmup_round=w + 1,
                moved_from_round=budget_round,
                selected_clients=clients,
                ae_examples=sum((r["num_seen"] for r in rows)),
                reconstruction_structure_loss=float(
                    np.average([r["loss"] for r in rows], weights=[r["num_seen"] for r in rows])
                ),
            )
        )
        state["warmup_done"] = w + 1
        checkpoint()
    for r in tqdm(
        range(state["round"] + 1, cfg.rounds + 1), desc=cfg.run_name, disable=phase == "smoke"
    ):
        fair.set_round(r)
        clients = fair.schedule[r - 1]
        old_routes = state["assignments"].copy()
        old_valid = state["previous_valid"].copy()
        train_ae = cfg.uses_latentfed and r <= cfg.rounds - warmup
        broadcast_router = server_router(state["ae_state"], cfg) if cfg.uses_latentfed else None
        rows = []
        for client in clients:
            set_seed(role_seed(cfg.seed, "local-stochastic", r, client))
            rows.append(
                train_one_client(
                    client,
                    state["cluster_models"][int(old_routes[client])],
                    state["ae_state"],
                    fair.train_loaders[client],
                    cfg,
                    channels,
                    size,
                    classes,
                    int(old_routes[client]),
                    state["prototypes"],
                    device,
                    r,
                    summary_loader=fair.assignment_loaders[client],
                    summary_router=broadcast_router,
                    train_ae=train_ae,
                )
            )
        diagnostics = {}
        geometry = None
        raw = np.full(len(clients), -1, dtype=int)
        aligned = raw.copy()
        if cfg.uses_latentfed:
            state["ae_state"] = weighted_average_states(
                [x["ae_state"] for x in rows], [x["num_train"] for x in rows]
            )
            router = server_router(state["ae_state"], cfg)
            geometry, inv, clu, summaries = make_geometry(
                [x["summary"] for x in rows], router, cfg, clients, state["summary_cache"], r
            )
            raw = cluster_summaries(geometry, cfg)
            aligned, centers, matched, state["next_cluster_id"] = align_clusters(
                raw,
                geometry,
                state["prototypes"].get("full_cluster", {}),
                state["next_cluster_id"],
                cfg,
            )
            routed = route_outliers(
                aligned,
                geometry,
                centers,
                old_valid[clients],
                old_routes[clients],
                state["cluster_models"],
            )
            state["assignments"][clients] = routed
            state["prototypes"] = update_prototypes(
                geometry, inv, clu, aligned, routed, raw, state["prototypes"], matched, cfg
            )
            diagnostics.update(router_diagnostics(router, summaries))
            diagnostics.update(mdfe_parameter_diagnostics(state["ae_state"], cfg))
            diagnostics.update(
                prototype_eligible_clients=(
                    int((raw >= 0).sum()) if cfg.exclude_noise_from_prototypes else len(raw)
                ),
                geometry_dimension=geometry.shape[1],
                prototype_dimension=clu.shape[1],
                matched_clusters=len(matched),
                raw_labels=raw.tolist(),
                aligned_labels=aligned.tolist(),
                routed_labels=routed.tolist(),
                clustering_geometry=geometry.tolist(),
            )
        else:
            state["assignments"].fill(0)
        current_valid = old_valid.copy()
        current_valid[clients] = aligned
        temporal = temporal_metrics(old_valid, current_valid, clients, state["previous_selected"])
        state["cluster_models"].update(aggregate_client_models(rows, state["assignments"]))
        state["cluster_models"] = {
            k: v
            for k, v in state["cluster_models"].items()
            if k in set(state["assignments"].tolist())
        }
        metrics, client_metrics = evaluate_table_iii_metrics(
            state["cluster_models"],
            state["assignments"],
            fair.evaluation_loaders,
            cfg,
            channels,
            classes,
            device,
            return_client_rows=True,
        )
        metrics["K_models"] = metrics.pop("K")
        metrics.update(clustering_metrics(geometry, raw))
        weights = [x["num_seen"] for x in rows]
        record = dict(
            round=r,
            selected_clients=clients,
            evaluation_split=cfg.evaluation_split,
            train_examples=int(sum(weights)),
            ae_examples=int(sum(weights)) if train_ae else 0,
            cald_kappa=cald_curriculum(r, cfg),
            client_metrics=client_metrics,
            assignments=state["assignments"].tolist(),
            train_loss=float(np.average([x["train_loss"] for x in rows], weights=weights)),
            train_acc=float(np.average([x["train_acc"] for x in rows], weights=weights)),
            **metrics,
            **diagnostics,
            **temporal,
        )
        for field in rows[0]:
            if field.startswith("loss_"):
                record[field] = float(np.average([x[field] for x in rows], weights=weights))
        if not record["WC"] - 1e-10 <= record["W-Acc"] <= record["BC"] + 1e-10:
            raise AssertionError("Invalid weighted classification metric")
        state["history"].append(record)
        state.update(round=r, previous_valid=current_valid, previous_selected=clients)
        if r % cfg.checkpoint_every == 0 or r == cfg.rounds or r == stop_after:
            checkpoint()
        if phase != "smoke" and (r == 1 or r % 10 == 0):
            print(
                f"round {r}: W-Acc={record['W-Acc']:.4f}, K={record['K']}, noise={record['noise_fraction']:.2f}"
            )
        if stop_after is not None and r >= stop_after and (r < cfg.rounds):
            return dict(interrupted=True, history=copy.deepcopy(state["history"]))
    expected = sum(
        (len(fair.train_parts[c]) * cfg.local_epochs for ids in fair.schedule for c in ids)
    )
    ae_examples = sum((x["ae_examples"] for x in state["history"])) + sum(
        (x["ae_examples"] for x in state["warmup_history"])
    )
    classifier_examples = sum((x["train_examples"] for x in state["history"]))
    if classifier_examples != expected or (cfg.uses_latentfed and ae_examples != expected):
        raise AssertionError("Local example budget changed")
    result = dict(
        config=asdict(cfg),
        phase=phase,
        manifest=fair.manifest,
        history=state["history"],
        warmup_history=state["warmup_history"],
        ae_examples=ae_examples,
        classifier_examples=classifier_examples,
        elapsed_this_session_seconds=time.monotonic() - started,
        seed_score=seed_summary(state["history"], cfg) if phase != "smoke" else None,
    )
    suite["runs"][key] = result
    suite["active"] = None
    if phase != "smoke":
        save_suite_checkpoint(suite, checkpoint_path)
    return result


## 26. Resumable LatentFed experiment

Restore the saved run or initialize a fresh suite, execute the method loop, and write its metrics.

In [ ]:
def save_suite_checkpoint(suite, checkpoint_path):
    if checkpoint_path is not None:
        atomic_checkpoint(checkpoint_path, suite)


def run_latentfed(cfg, output, smoke=False, resume=True, stop_after=None):
    folder = _run_folder(output, "latentfed", cfg, smoke)
    checkpoint = folder / "checkpoint.pt"
    suite = _state_or_fresh(checkpoint, cfg, "latentfed", resume)
    for key, value in fresh_suite().items():
        suite.setdefault(key, value)
    if suite["frozen"] is None:
        frozen = dict(
            config=asdict(cfg),
            datasets=list(PAPER_DATASETS),
            alphas=list(PAPER_ALPHAS),
            seeds=list(PAPER_SEEDS),
            final_round_window=[91, 100],
            std_ddof=1,
        )
        frozen["digest"] = config_digest(frozen)
        suite["frozen"] = frozen
    bundle = synthetic_bundle(cfg.dataset) if smoke else None
    try:
        result = run_experiment(
            cfg,
            phase="smoke" if smoke else "final",
            suite=suite,
            dataset_bundle=bundle,
            stop_after=stop_after,
            checkpoint_path=checkpoint,
        )
    finally:
        atomic_checkpoint(checkpoint, suite)
    complete = not result.get("interrupted", False)
    history = result["history"]
    manifest = result["manifest"] if complete else suite["active"]["manifest"]
    return write_result(
        folder,
        cfg,
        history,
        manifest,
        "latentfed",
        smoke,
        complete,
        _score(history, cfg, "latentfed") if complete and not smoke else None,
        extra=dict(execution="serial single device", ae_architecture=cfg.ae_architecture),
    )


## 27. Experiment driver

Run the selected dataset/alpha/seed/variant batch and refresh the ZIP after each experiment.

In [ ]:
def run_grid(
    datasets=PAPER_DATASETS,
    alphas=PAPER_ALPHAS,
    seeds=PAPER_SEEDS,
    variants=("full",),
    output="outputs",
    smoke=False,
    resume=True,
    include_checkpoints=False,
    allow_cpu=False,
):
    if not smoke and not torch.cuda.is_available() and not allow_cpu:
        raise RuntimeError(
            "Enable a Kaggle GPU for paper mode, or set SMOKE_TEST=True for the short CPU check."
        )
    if smoke:
        torch.set_num_threads(1)
    plan = experiment_plan(datasets, alphas, seeds, variants, smoke=smoke)
    output = Path(output)
    mode_root = output / ("smoke" if smoke else "paper") / "latentfed"
    mode_root.mkdir(parents=True, exist_ok=True)
    json_write(
        mode_root / "experiment_plan.json",
        dict(
            method="latentfed",
            mode="smoke" if smoke else "paper",
            experiments=plan,
            clients=20,
            clients_per_round=20,
            rounds=2 if smoke else 100,
            local_epochs=1,
            batch_size=32,
        ),
    )
    folders = []
    try:
        for item in plan:
            cfg = make_config(**item, method="latentfed", data_root=output / "data", smoke=smoke)
            print(
                f"Running {cfg.dataset} / {cfg.variant} / alpha={cfg.alpha} / seed={cfg.seed} / rounds={cfg.rounds}",
                flush=True,
            )
            folders.append(run_latentfed(cfg, output, smoke=smoke, resume=resume))
            aggregate_runs(mode_root)
            export_zip(mode_root, include_checkpoints)
    finally:
        archive = export_zip(mode_root, include_checkpoints)
    return dict(runs=folders, archive=archive, summary=aggregate_runs(mode_root)[0])


## 28. Plotting helper

Preview the last completed run per dataset/variant. Recorded values remain unsmoothed, and synthetic smoke figures are labelled.

In [ ]:
def plot_run_preview(paths):
    """One last completed run per dataset/variant, clearly scoped as a preview."""
    chosen = {}
    for path in map(Path, paths):
        metadata = json.loads((path / "manifest.json").read_text())
        chosen[metadata["dataset"], metadata["variant"]] = (path, metadata)
    colors = {"cifar10": "#245B8A", "emnist": "#B76B18"}
    styles = {
        "full": ("-", "o"),
        "no_mdfe": ("--", "s"),
        "no_cald": (":", "^"),
        "ae_only": ("-.", "D"),
    }
    figure, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for (dataset, variant), (path, metadata) in chosen.items():
        frame = pd.read_csv(path / "round_metrics.csv")
        linestyle, marker = styles[variant]
        appearance = dict(
            color=colors[dataset],
            linestyle=linestyle,
            marker=marker,
            markersize=3,
            markevery=max(1, len(frame) // 8),
            linewidth=1.5,
            label=f"{dataset}: {variant}",
        )
        axes[0].plot(frame["round"], 100 * frame["W-Acc"], **appearance)
        axes[1].plot(frame["round"], frame["train_loss"], **appearance)
    axes[0].set(xlabel="Communication round", ylabel="Weighted test accuracy (%)", ylim=(0, 100))
    axes[1].set(xlabel="Communication round", ylabel="Recorded local training objective")
    for axis in axes:
        axis.grid(alpha=0.2)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="lower center", ncol=min(4, len(labels)), fontsize=8)
    smoke = all((m["mode"] == "smoke" for _, m in chosen.values()))
    figure.suptitle(
        "Synthetic smoke checks — not paper results"
        if smoke
        else "Last completed seed/configuration per dataset and variant"
    )
    figure.tight_layout(rect=(0, 0.14, 1, 0.98))
    return figure


## 29. Inspect the plan and models

Review the selected grid and model class before training starts.

In [ ]:
PLAN = experiment_plan(DATASETS, ALPHAS, SEEDS, VARIANTS, smoke=SMOKE_TEST)
display(pd.DataFrame(PLAN).head(12))
print("Experiments in this batch:", len(PLAN))
model_rows = []
for variant in VARIANTS:
    cfg = make_config(DATASETS[0], ALPHAS[0], SEEDS[0], variant, method=METHOD, smoke=SMOKE_TEST)
    channels, size, classes = (3, 32, 10) if DATASETS[0] == "cifar10" else (1, 28, 47)
    model = (
        make_autoencoder(channels, size, cfg)
        if METHOD == "latentfed"
        else SimpleCNN(channels, classes)
    )
    model_rows.append(
        dict(
            variant=variant,
            model=type(model).__name__,
            parameters=sum(p.numel() for p in model.parameters()),
            convolution_layers=sum(
                isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)) for m in model.modules()
            ),
        )
    )
display(pd.DataFrame(model_rows))


## 30. Run the selected experiments

Real-data mode uses the full 100-round budget and a CUDA GPU.

In [ ]:
RESULT = run_grid(
    datasets=DATASETS,
    alphas=ALPHAS,
    seeds=SEEDS,
    variants=VARIANTS,
    output=OUTPUT_ROOT,
    smoke=SMOKE_TEST,
    resume=RESUME,
    include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP,
)
print("Completed run folders:", len(RESULT["runs"]))


## 31. Results preview

Curves show the last completed run per dataset/variant. Five-seed CSVs are the statistical output; synthetic smoke figures are labelled.

In [ ]:
status = []
for folder in RESULT["runs"]:
    manifest = json.loads((folder / "manifest.json").read_text())
    status.append(
        {
            key: manifest[key]
            for key in [
                "method",
                "variant",
                "dataset",
                "alpha",
                "seed",
                "mode",
                "completed_rounds",
                "publication_eligible",
            ]
        }
    )
display(pd.DataFrame(status))
if RESULT["summary"].empty:
    print("No complete five-seed real-data group in this output folder yet.")
else:
    display(RESULT["summary"])
figure = plot_run_preview(RESULT["runs"])
display(figure)
plt.close(figure)


## 32. Download and resume

The ZIP is also available in Kaggle Output. It contains settings, manifests, round/client metrics, diagnostics and coverage. Set `INCLUDE_CHECKPOINTS_IN_ZIP=True` before exporting to carry checkpoints to another session. Restore the output hierarchy and compatible runtime to resume. Reusing a completed run does not add another seed.

In [ ]:
mode_root = OUTPUT_ROOT / ("smoke" if SMOKE_TEST else "paper") / METHOD
DOWNLOAD_ZIP = export_zip(mode_root, include_checkpoints=INCLUDE_CHECKPOINTS_IN_ZIP)
link_path = (
    str(DOWNLOAD_ZIP)
    if Path("/kaggle/working").is_dir()
    else os.path.relpath(DOWNLOAD_ZIP, Path.cwd())
)
display(FileLink(link_path))
print("Download ZIP:", DOWNLOAD_ZIP)
